# Player Stats, Advanced Metrics, and Leaderboards

Division I player seasons, 2021-2025.

**The cache stores counting statistics only.** Every rate and advanced
statistic is computed when you read it, from those counts plus league constants
this package derives from its own NCAA team data — so they can never fall out
of step with the data underneath.

Covers: `list_available_years`, `list_players`, `list_batters`, `list_pitchers`,
`player_seasons`, `batting_stat`, `pitching_stat`, `get_player_rows`,
`load_player_frame`, `top_players`, `leaderboard`, `stat_direction`,
`qualification_rules`, `cwoba`, `cwraa`, `cwrc`, `cwrc_plus`, `cwsb`, `cspd`,
`cfip`, `clob_pct`, `league_constants`, `seasons_with_constants`

In [1]:
from ncaa_bbStats import *

## Finding players

In [2]:
print("batting seasons:", list_available_years("batting", "qualified"))
print("pitching seasons:", list_available_years("pitching", "qualified"))

batting seasons: [2021, 2022, 2023, 2024, 2025, 2026]
pitching seasons: [2021, 2022, 2023, 2024, 2025, 2026]


In [3]:
# qualified = met the playing-time minimum; noMin = everyone
print("2025 batters, no minimum:", len(list_batters(year=2025)))
print("2025 batters, qualified :", len(list_batters("qualified", year=2025)))
print("2025 pitchers, qualified:", len(list_pitchers("qualified", year=2025)))

# Narrow by team
print("\nNortheastern batters 2025:", list_batters(year=2025, team_substr="NE")[:5])

2025 batters, no minimum: 5351
2025 batters, qualified : 2377
2025 pitchers, qualified: 1067

Northeastern batters 2025: ['Alex Lane', 'Antonio Avila', 'Billy Ham', 'Cael Frost', 'Cameron Maldonado']


In [4]:
print(qualification_rules("batting", 2025))
print(qualification_rules("pitching", 2025))

{'stat_type': 'batting', 'year': 2025, 'per_game': 3.1, 'basis': 'plate appearances', 'typical_season_games': 56, 'typical_threshold': 173.6}
{'stat_type': 'pitching', 'year': 2025, 'per_game': 1.0, 'basis': 'innings pitched', 'typical_season_games': 56, 'typical_threshold': 56.0}


In [5]:
# list_players is the general form behind list_batters / list_pitchers
print(list_players("pitching", "qualified", year=2025)[:5])

print("\nAiven Cabral pitched in:", player_seasons("pitching", "noMin", "Aiven Cabral"))

['A.J. Colarusso', 'AJ Ciscar', 'AJ Kostic', 'AJ Petraitis', 'Aaron Boster']



Aiven Cabral pitched in: [2023, 2024, 2025]


## Reading a statistic

In [6]:
# Counting stats
print("Jack Goodman 2025 HR:", batting_stat("Jack Goodman", "hr", year=2025))

# Rate stats are computed on read -- no column holds them
for stat in ["avg", "obp", "slg", "ops", "iso", "babip", "bb%", "k%"]:
    print(f"  {stat:6s} {batting_stat('Jack Goodman', stat, year=2025):.4f}")

Jack Goodman 2025 HR:

 10.0
  avg    0.3350
  obp    0.4060
  slg    0.5468
  ops    0.9528
  iso    0.2118
  babip  0.3919
  bb%    0.1111
  k%     0.2094


In [7]:
for stat in ["so", "era", "whip", "k/9", "bb/9", "k-bb%"]:
    print(f"  {stat:6s} {pitching_stat('Aiven Cabral', stat, year=2025):.4f}")

  so     74.0000
  era    2.9216
  whip   1.0299
  k/9    7.4552
  bb/9   1.4104
  k-bb%  0.1667


### Multi-season figures are rebuilt, not averaged

Asking for a stat without a year aggregates the player's whole career.
Counting stats are summed; **rates are recomputed from the summed components**.
A pitcher who threw to a 3.00 ERA one year and 5.00 the next did not have an
8.00 ERA.

In [8]:
rows = get_player_rows("pitching", "noMin", "Aiven Cabral",
                       include_columns=["year", "ip", "er", "era", "so"])
for row in rows:
    print(f"  {row['year']}  {row['ip']:>6} IP  {row['er']:>3} ER  ERA {row['era']:.2f}")

print(f"\n  career IP  {pitching_stat('Aiven Cabral', 'ip')}  (true innings)")
print(f"  career ER  {pitching_stat('Aiven Cabral', 'er')}")
print(f"  career ERA {pitching_stat('Aiven Cabral', 'era'):.2f}  "
      f"<- 9 x ER / IP, not the mean of the three seasons")

  2023    83.2 IP   24 ER  ERA 2.58
  2024    42.0 IP   33 ER  ERA 7.07
  2025    89.1 IP   29 ER  ERA 2.92

  career IP  215.0  (true innings)
  career ER  86.0
  career ERA 3.60  <- 9 x ER / IP, not the mean of the three seasons


In [9]:
# The whole table, with every derived column attached
frame = load_player_frame("batting", "qualified")
print(frame.shape)
print(list(frame.columns))

(13910, 42)
['player_id', 'name', 'team', 'team name', 'division', 'year', 'age', 'g', 'pa', 'ab', 'h', '2b', '3b', 'hr', 'r', 'rbi', 'bb', 'so', 'hbp', 'sf', 'sh', 'gdp', 'sb', 'cs', 'qualified', '1b', 'tb', 'avg', 'obp', 'slg', 'ops', 'iso', 'bb%', 'k%', 'bb/k', 'babip', 'cwoba', 'cwraa', 'cwrc', 'cwrc+', 'cwsb', 'cspd']


## Advanced metrics

Metrics prefixed `c` are college-calibrated: built the same way as the familiar
sabermetric statistics, but with league constants regressed from NCAA play
rather than borrowed from elsewhere.

In [10]:
for stat in ["cwoba", "cwrc+", "cwraa", "cwrc", "cwsb", "cspd"]:
    print(f"  {stat:8s} {batting_stat('Jack Goodman', stat, year=2025):8.3f}")

print()
for stat in ["cfip", "clob%", "e-cf"]:
    print(f"  {stat:8s} {pitching_stat('Aiven Cabral', stat, year=2025):8.3f}")

  cwoba       0.427
  cwrc+     119.458
  cwraa       7.747
  cwrc       47.560
  cwsb        0.850
  cspd        5.407

  cfip        3.750
  clob%       0.728
  e-cf       -0.828


In [11]:
# The same metrics computed from a stat line directly
line = {"ab": 203, "h": 68, "2b": 17, "3b": 1, "hr": 10,
        "bb": 26, "hbp": 5, "sf": 0, "pa": 234, "so": 49, "sb": 9, "cs": 2, "r": 51}

print("cwoba    ", round(cwoba(line, 2025, division=1), 4))
print("cwraa    ", round(cwraa(line, 2025, division=1), 2))
print("cwrc     ", round(cwrc(line, 2025, division=1), 2))
print("cwrc_plus", round(cwrc_plus(line, 2025, division=1), 1))
print("cwsb     ", round(cwsb(line, 2025, division=1), 3))
print("cspd     ", round(cspd(line), 2))

cwoba     0.4488
cwraa     11.87
cwrc      51.68
cwrc_plus 129.8
cwsb      0.477
cspd      5.08


In [12]:
pitcher = {"ip": 89.1, "hr": 6, "bb": 29, "so": 74, "hbp": 6,
           "h": 63, "r": 34, "er": 29, "tbf": 361}
print("cfip    ", round(cfip(pitcher, 2025, division=1), 3))
print("clob_pct", round(clob_pct(pitcher), 4))

cfip     4.657
clob_pct 0.7143


### The constants behind them

Run values are regressed from the packaged NCAA team-stats cache, per season and
division. `cwrc_plus` applies **no park adjustment** — NCAA park data is not
public — so hitters at extreme-altitude programs are flattered.

In [13]:
constants = league_constants(2025, division=1)
print("run values, D-I 2025:")
for event in ["1b", "2b", "3b", "hr", "bb", "hbp", "sb", "cs"]:
    print(f"  {event:4s} {constants['w_' + event]:+.3f}")
print(f"\n  league OBP    {constants['lg_obp']:.4f}")
print(f"  league R/PA   {constants['lg_r_pa']:.4f}")
print(f"  park factor   {constants['park_factor']}  (not modelled)")
print(f"  fit R2        {constants['r2']}")

run values, D-I 2025:
  1b   +0.953
  2b   +1.274
  3b   +1.725
  hr   +1.995
  bb   +0.753
  hbp  +0.799
  sb   +0.322
  cs   -0.447

  league OBP    0.3857
  league R/PA   0.1701
  park factor   1.0  (not modelled)
  fit R2        0.9606


In [14]:
pitching_constants = league_constants(2025, division=1, kind="pitching")
print(f"league ERA    {pitching_constants['lg_era']:.3f}")
print(f"cFIP constant {pitching_constants['cfip_constant']:.3f}")

# Seasons too sparse to fit return None rather than a fabricated number
print("\nD-I seasons with constants:", seasons_with_constants(1))
print("D-III:", seasons_with_constants(3))
print("\n2005 is too sparse ->", cwoba(line, 2005, division=1))

league ERA    6.171
cFIP constant 4.265

D-I seasons with constants: [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
D-III: [2009, 2010, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

2005 is too sparse -> None


## Leaderboards

`leaderboard` takes the sort direction from the statistic, so a top-ERA list
contains good pitchers. `top_players` is the older function and always sorts
descending.

In [15]:
print("top_players sorts descending unconditionally:")
for row in top_players("pitching", "era", 3, 2025):
    print(f"  {row['name']:22s} {row['value']:.2f}   <- the WORST ERAs")

print("\nleaderboard picks the direction:")
for row in leaderboard("era", stat_type="pitching", year=2025, n=3, min_ip=50):
    print(f"  {row['name']:22s} {row['value']:.3f}")

top_players sorts descending unconditionally:
  Jermel Ford            13.99   <- the WORST ERAs
  Nick Espinola          13.50   <- the WORST ERAs
  Luke Sanders           13.11   <- the WORST ERAs

leaderboard picks the direction:
  Jack Ohman             1.344
  Jonathan Gonzalez      1.826
  Dylan Volantis         1.941


In [16]:
for stat, kind in [("era", "pitching"), ("so", "pitching"), ("so", "batting"),
                   ("hr", "batting"), ("whip", "pitching"), ("cwrc+", "batting")]:
    print(f"  {stat:6s} ({kind:8s}) -> {stat_direction(stat, kind)} is better")

  era    (pitching) -> lower is better
  so     (pitching) -> higher is better
  so     (batting ) -> lower is better
  hr     (batting ) -> higher is better
  whip   (pitching) -> lower is better
  cwrc+  (batting ) -> higher is better


In [17]:
# Filter by conference, and carry extra columns through
for row in leaderboard("cwrc+", year=2025, conference="SEC", n=8,
                       include=["hr", "ops"]):
    print(f"  {row['name']:22s} {row['team']:6s} "
          f"cwRC+ {row['value']:6.1f}  {row['hr']:.0f} HR  OPS {row['ops']:.3f}")

  Ryland Zaborowski      UGA    cwRC+  178.2  17 HR  OPS 1.288
  Andrew Fischer         TENN   cwRC+  169.5  25 HR  OPS 1.257
  Ike Irish              AUB    cwRC+  160.8  19 HR  OPS 1.179
  Robbie Burnett         UGA    cwRC+  157.9  20 HR  OPS 1.169
  Noah Sullivan          MSST   cwRC+  152.4  15 HR  OPS 1.120
  Ace Reese              MSST   cwRC+  148.2  21 HR  OPS 1.140
  Gavin Kilen            TENN   cwRC+  147.9  15 HR  OPS 1.112
  Wehiwa Aloy            ARK    cwRC+  146.0  21 HR  OPS 1.107


In [18]:
# Team names accept any spelling
print("by name   :", [r["name"] for r in leaderboard("hr", year=2025, team="Auburn", n=3)])
print("by acronym:", [r["name"] for r in leaderboard("hr", year=2025, team="AUB", n=3)])

by name   : ['Ike Irish', 'Cooper McMurray', 'Chris Rembert']
by acronym: ['Ike Irish', 'Cooper McMurray', 'Chris Rembert']


In [19]:
# Career leaderboards aggregate first, rebuilding rates from summed components
print("Career home runs, 2021-2025:")
for row in leaderboard("hr", per="career", n=5, qualifier="noMin"):
    print(f"  {row['name']:22s} {row['value']:5.0f} HR over {row['seasons']} seasons")

print("\nCareer ERA (minimum 150 innings):")
for row in leaderboard("era", stat_type="pitching", per="career",
                       n=5, min_ip=150, qualifier="noMin"):
    print(f"  {row['name']:22s} {row['value']:.3f} over {row['seasons']} seasons")

Career home runs, 2021-2025:


  Jac Caglianone            75 HR over 3 seasons
  Tommy White               75 HR over 3 seasons
  Brock Wilken              71 HR over 3 seasons
  Hunter Hines              70 HR over 4 seasons
  Jace LaViolette           68 HR over 3 seasons

Career ERA (minimum 150 innings):
  Paul Skenes            2.183 over 3 seasons
  Andrew Taylor          2.473 over 2 seasons
  Jackson Flora          2.491 over 3 seasons
  Zander Sechrist        2.516 over 4 seasons
  Izaak Martinez         2.578 over 4 seasons
